# TOTNet Tennis Inference (Colab)
Tracks a tennis ball in a video, saves annotated frames and an MP4.

In [ ]:
# 1) Check GPU
import torch, os, sys, platform
print('CUDA available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Python:', sys.version)
print('OS:', platform.platform())

In [ ]:
# 2) Install dependencies (PyTorch 2.4.1 CUDA 12.1 wheel matches Colab GPUs)
%pip install -q torch==2.4.1+cu121 torchvision==0.19.1+cu121 torchaudio==2.4.1+cu121 --extra-index-url https://download.pytorch.org/whl/cu121
%pip install -q easydict matplotlib opencv-python scikit-learn cython pycocotools tqdm scipy ninja tensorboard ptflops
!apt-get -y install ffmpeg

In [ ]:
# 3) Clone TOTNet repo and move inside
!git clone https://github.com/AugustRushG/TOTNet.git
%cd TOTNet/src

In [ ]:
# 4) Download pretrained weights from Zenodo and extract
import os, zipfile, glob, urllib.request
os.makedirs('../weights', exist_ok=True)
url = 'https://zenodo.org/records/11244990/files/weight%20and%20texts.zip?download=1'  # Zenodo v2
zip_path = '/content/weight_and_texts.zip'
urllib.request.urlretrieve(url, zip_path)
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('../weights')
pths = glob.glob('../weights/**/*.pth', recursive=True) + glob.glob('../weights/*.pth')
assert len(pths) > 0, 'No .pth weights found inside the zip.'
WEIGHT_PATH = pths[0]
print('Using weight:', WEIGHT_PATH)

In [ ]:
# 5) Provide a tennis video
# Option A: upload manually via the Colab file browser, then set VIDEO_PATH below
# Option B: download a sample clip (replace the URL with your own)
VIDEO_PATH = '/content/input.mp4'
sample_url = 'https://storage.googleapis.com/gtv-videos-bucket/sample/ForBiggerJoyrides.mp4'  # placeholder demo clip
!wget -O $VIDEO_PATH $sample_url
print('Video ready at', VIDEO_PATH)

In [ ]:
# 6) Run inference and save annotated frames + MP4
import os, subprocess, json, textwrap, sys
SAVE_NAME = 'colab_tennis_demo'
OUTPUT_ROOT = '/content/totnet_outputs'
os.makedirs(OUTPUT_ROOT, exist_ok=True)

cmd = [
    'python', 'demo.py',
    '--model_choice', 'motion_light',
    '--dataset_choice', 'tt',            # uses Video_Loader for raw videos
    '--video_path', VIDEO_PATH,
    '--pretrained_path', WEIGHT_PATH,
    '--num_frames', '5',
    '--img_size', '288', '512',
    '--gpu_idx', '0',
    '--save_demo_output',
    '--output_format', 'video',
    '--working-dir', '/content',
    '--saved_fn', SAVE_NAME
]

print('Running:\n', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
print(result.stderr)

# Locate result paths
frames_dir = f'/content/results/demo/{SAVE_NAME}/frame'
video_path = f'/content/results/demo/{SAVE_NAME}/result.mp4'
print('Frames dir:', frames_dir)
print('Video path:', video_path)
assert os.path.isfile(video_path), 'Output video not found—check logs above.'

In [ ]:
# 7) Preview the annotated video inline
from IPython.display import Video, HTML
display(Video(video_path, embed=True, width=720))

In [ ]:
# 8) Optional: save to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp $video_path /content/drive/MyDrive/